<a href="https://colab.research.google.com/github/prince71510-sys/Deep-Learning-Projects/blob/main/CAB_Temperature_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# --- 1. DATA PREPARATION ---
# Feature Engineering for ML models
df['temp_lag1'] = df['temp'].shift(1)
df['hum_lag1'] = df['humidity'].shift(1)
df_clean = df.dropna()

# Split: 80% Train, 20% Test (Chronological)
split = int(len(df_clean) * 0.8)
train, test = df_clean.iloc[:split], df_clean.iloc[split:]

# --- 2. TRAINING THE MODELS ---

# A. SARIMAX
model_sarimax = SARIMAX(train['temp'], exog=train['humidity'], order=(1,1,1)).fit(disp=False)
pred_sarimax = model_sarimax.forecast(steps=len(test), exog=test['humidity'])

# B. RANDOM FOREST
rf = RandomForestRegressor(n_estimators=100)
rf.fit(train[['temp_lag1', 'hum_lag1']], train['temp'])
pred_rf = rf.predict(test[['temp_lag1', 'hum_lag1']])

# C. XGBOOST
model_xgb = xgb.XGBRegressor(n_estimators=100)
model_xgb.fit(train[['temp_lag1', 'hum_lag1']], train['temp'])
pred_xgb = model_xgb.predict(test[['temp_lag1', 'hum_lag1']])

# D. LSTM (Requires Scaling)
scaler = MinMaxScaler()
scaled_train = scaler.fit_transform(train[['temp', 'humidity']])
scaled_test = scaler.transform(test[['temp', 'humidity']])

# Simple helper to reshape for LSTM (1-day lag)
X_l, y_l = scaled_train[:-1], scaled_train[1:, 0]
X_t, y_t = scaled_test[:-1], scaled_test[1:, 0]

model_lstm = Sequential([LSTM(50, input_shape=(1, 2)), Dense(1)])
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_l.reshape(-1, 1, 2), y_l, epochs=10, verbose=0)

pred_lstm_scaled = model_lstm.predict(X_t.reshape(-1, 1, 2))
# Inverse scale back to temperature
pred_lstm = pred_lstm_scaled * (df['temp'].max() - df['temp'].min()) + df['temp'].min()

# --- 3. EVALUATION & VISUALIZATION ---

results = {
    "SARIMAX": mean_absolute_error(test['temp'], pred_sarimax),
    "Random Forest": mean_absolute_error(test['temp'], pred_rf),
    "XGBoost": mean_absolute_error(test['temp'], pred_xgb),
    "LSTM": mean_absolute_error(test['temp'][1:], pred_lstm)
}

# Plotting the Leaderboard

plt.figure(figsize=(10, 5))
plt.bar(results.keys(), results.values(), color=['#3498db', '#e74c3c', '#2ecc71', '#f1c40f'])
plt.ylabel('Mean Absolute Error (Lower is Better)')
plt.title('AC Temperature Prediction: Model Tournament')
plt.show()

print("Tournament Results (MAE):", results)